# COMP0258 — ReMDM Discrete Diffusion Planning in Craftax

> **Self-contained Colab notebook.** Loads pre-trained checkpoints from a public
> HuggingFace repo and demonstrates live inference, agent visualisation and the
> ReMDM denoising loop. No training is performed inside the notebook.

**How to use this notebook**

1. Open in Google Colab (preferably with a GPU runtime).
2. Run all cells top-to-bottom.
3. To test on **unseen inputs**, edit the constants in the *Configuration*
   cell (`SEED`, `ENV_NAME`, `EVAL_STEPS`, `EVAL_NUM_ENVS`, `DIFFUSION_STEPS_EVAL`)
   and re-run from there. Every Craftax seed produces a procedurally generated
   world the agent has never seen.

**Submission compliance**

- Cell 1 downloads everything from a single public HuggingFace repo
  (`HF_REPO_ID`) — no authentication required.
- The pre-trained checkpoint is loaded; no training happens here.
- Live inference (Cells 5–7) demonstrates that reported numbers reproduce.
- Pre-computed ablation figures (Cells 9–10) demonstrate the research finding.


In [ ]:
# =============================================================================
# CONFIGURATION — marker can edit any of these and re-run from here
# =============================================================================

# Public HuggingFace repo containing src/, Craftax_Baselines/, configs/,
# checkpoints/ and pre-computed ablation outputs.
HF_REPO_ID = "TODO_HF_REPO_ID"
LOCAL_DIR = "remdm-craftax"

# --- Reproducibility & evaluation knobs ---
SEED = 42

# "Craftax-Classic-Symbolic-v1": 22 achievements, 17 actions  (DAgger checkpoint)
# "Craftax-Symbolic-v1":         65 achievements, 43 actions  (PPO expert only)
ENV_NAME = "Craftax-Classic-Symbolic-v1"

EVAL_STEPS = 2000          # Per-env step budget for live diffusion eval
EVAL_NUM_ENVS = 16         # Parallel environments (lower if Colab OOMs)
DIFFUSION_STEPS_EVAL = 10  # Reverse denoising steps at inference time


## 1. Project overview

**Problem.** Plan action sequences in *Craftax*, a JAX-accelerated procedurally
generated open-world survival game (a Crafter reimplementation extended with
NetHack-like mechanics).

**Approach.** A bidirectional **denoising transformer** generates a
`plan_horizon = 32` action plan by iteratively denoising masked discrete tokens
(MDLM / **ReMDM** — Wang et al.) conditioned on the current symbolic
observation. At inference, MPC executes one action per plan and re-plans every
step with **historical inpainting**: positions `0 .. hist_len - 1` are locked to
the actions actually taken.

**Pipeline (offline, before this notebook).**
```
[1] PPO-RNN expert        (Craftax_Baselines/ppo_rnn.py)
[2] Offline behaviour cloning   (main.py --mode offline)
[3] Online DAgger fine-tuning   (main.py --mode online)
[4] RL fine-tuning ablation suite — 25 ablations
```

**Research question.** Can RL fine-tuning improve a DAgger-pretrained masked
diffusion planner?

**Headline finding.** DAgger faithfully imitates the PPO expert, but **no RL
ablation** (25 tested across regularisation, optimisation, data and capacity)
meaningfully improves on the DAgger checkpoint. This matches the parallel
finding from our MiniHack codebase, indicating the obstruction is
framework-independent.

The notebook demonstrates the three things the marker needs to verify:

| Cell | Demonstrates |
|---|---|
| 5  | Live inference numbers reproduce reported results |
| 6  | The agent meaningfully interacts with Craftax (rewards, achievements) |
| 7  | The ReMDM iterative-unmasking sampler in action |
| 8  | DAgger ≈ PPO expert (live PPO eval) |
| 9–10 | Pre-computed ablation results — RL doesn't help |


In [ ]:
# =============================================================================
# Install pinned dependencies, download the HF repo, set up sys.path
# =============================================================================

import importlib
import os
import subprocess
import sys


def _pip(*pkgs: str) -> None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *pkgs],
        stdout=subprocess.DEVNULL,
    )


# Core dependencies (versions match pyproject.toml).
_pip(
    "huggingface_hub>=1.9.1",
    "craftax>=1.5.0",
    "flax>=0.12.6",
    "optax>=0.2.8",
    "orbax-checkpoint>=0.5",
    "distrax>=0.1.7",
    "chex>=0.1.91",
    "polars>=1.39.3",
    "orjson>=3.11.8",
    "pyyaml>=6.0",
)

# JAX: keep Colab's preinstalled GPU build if present, otherwise install CPU.
try:
    import jax  # noqa: F401
    if jax.default_backend() == "gpu":
        pass
    else:
        raise RuntimeError("re-install needed")
except Exception:
    _pip("jax>=0.9.2")
    importlib.invalidate_caches()
    import jax  # noqa: F401

import craftax  # noqa: F401

backend = jax.default_backend()
device = jax.devices()[0]
print(f"JAX {jax.__version__} | backend={backend} | device={device}")
print(f"Craftax {craftax.__version__}")
if backend != "gpu":
    print(
        "WARNING: JAX is running on CPU. Inference will be ~10x slower than GPU."
        "\n         In Colab, switch to a GPU runtime via Runtime -> Change runtime type."
    )

# ----------------------------------------------------------------------------
# Pull the project tree (code, configs, checkpoints, pre-computed results)
# from a single public HuggingFace repo. No authentication required.
# ----------------------------------------------------------------------------
from huggingface_hub import snapshot_download

snapshot_path = snapshot_download(repo_id=HF_REPO_ID, local_dir=LOCAL_DIR)
print(f"Snapshot downloaded to: {snapshot_path}")

# Both the parent (for `import src.planners...`) and the Craftax_Baselines/
# subdir (for `from wrappers import ...` inside Craftax_Baselines/ppo_rnn.py)
# must be on sys.path. main.py does the same thing.
for p in (snapshot_path, os.path.join(snapshot_path, "Craftax_Baselines")):
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(snapshot_path)
print(f"cwd: {os.getcwd()}")


In [ ]:
# =============================================================================
# Load the pre-trained DAgger diffusion checkpoint
# =============================================================================

import json

import jax
import jax.numpy as jnp
from craftax.craftax_env import make_craftax_env_from_name

from src.planners.model import build_model, load_checkpoint, make_apply_fns

# Checkpoint inventory bundled with the HF repo.
DIFFUSION_OFFLINE_CKPT = (
    "checkpoints/offline/Craftax-Classic-Symbolic-v1-OfflineDiffusion-BC-100M"
)
DIFFUSION_ONLINE_CKPT = (
    "checkpoints/online/Craftax-Classic-Symbolic-v1-OnlineDiffusion-DAgger-50M"
)
PPO_CKPT = {
    "Craftax-Classic-Symbolic-v1": (
        "checkpoints/ppo_agents/Craftax-Classic-Symbolic-v1-PPO_RNN-1000M"
    ),
    "Craftax-Symbolic-v1": (
        "checkpoints/ppo_agents/Craftax-Symbolic-v1-PPO_RNN-1000M"
    ),
}

# We pin to the DAgger (online) checkpoint — that is the headline result.
# The architecture hyperparameters are stored alongside the offline checkpoint
# in `resume_metadata.json` (online and offline share the same architecture).
with open(os.path.join(DIFFUSION_OFFLINE_CKPT, "resume_metadata.json")) as f:
    META = json.load(f)
ARCH_CFG = META["config_snapshot"]

# The diffusion checkpoints are trained on Classic Craftax. The denoiser's
# action head dimension is fixed at training time (num_actions = 17 for Classic),
# so we cannot transfer it to Full Craftax (43 actions). PPO covers both.
DIFFUSION_ENV_NAME = "Craftax-Classic-Symbolic-v1"

env_init = make_craftax_env_from_name(DIFFUSION_ENV_NAME, auto_reset=True)
env_params_init = env_init.default_params
NUM_ACTIONS = int(env_init.action_space(env_params_init).n)
OBS_DIM = int(env_init.observation_space(env_params_init).shape[0])
PLAN_HORIZON = int(ARCH_CFG["PLAN_HORIZON"])

print(f"DiffusionEnv : {DIFFUSION_ENV_NAME}")
print(f"  obs_dim    : {OBS_DIM}")
print(f"  num_actions: {NUM_ACTIONS}")
print(f"Architecture : d_model={ARCH_CFG['D_MODEL']} n_layers={ARCH_CFG['N_LAYERS']} "
      f"n_heads={ARCH_CFG['N_HEADS']} plan_horizon={PLAN_HORIZON}")

model = build_model(ARCH_CFG, NUM_ACTIONS)
apply_eval, _ = make_apply_fns(model)
diffusion_params = load_checkpoint(
    model,
    jax.random.PRNGKey(SEED),
    OBS_DIM,
    PLAN_HORIZON,
    DIFFUSION_ONLINE_CKPT,
)

n_params = sum(int(p.size) for p in jax.tree.leaves(diffusion_params))
print(f"Loaded DAgger checkpoint: {n_params / 1e6:.2f}M parameters")


In [ ]:
# =============================================================================
# CELL 5 (PRIORITY) — live inference on `EVAL_NUM_ENVS` parallel envs
# =============================================================================
#
# This calls the *exact* production inference code path used by
# `python main.py --mode inference`. The marker can change ENV_NAME / SEED /
# EVAL_STEPS in Cell 1 to test on entirely fresh procedurally generated worlds.
#
# Reported numbers (DAgger online checkpoint, EVAL_STEPS=10 000, 32 envs,
# Craftax-Classic-Symbolic-v1):
#     mean episode return ≈ 10.4   (compare with PPO expert ≈ 10.4)
#     score ceiling: 22 (one point per achievement)
# -----------------------------------------------------------------------------

from src.planners.inference import run_inference

if "Classic" not in ENV_NAME:
    print(
        f"ENV_NAME={ENV_NAME!r}: no diffusion checkpoint exists for Full Craftax\n"
        f"(action vocabularies differ: 17 vs 43). Skipping diffusion eval — see\n"
        f"Cell 8 for the matching PPO expert evaluation on this environment."
    )
else:
    inference_cfg = {
        # Architecture (read from checkpoint metadata)
        **{k: v for k, v in ARCH_CFG.items() if k.isupper()},
        # Marker-controlled
        "ENV_NAME": ENV_NAME,
        "SEED": SEED,
        "EVAL_STEPS": EVAL_STEPS,
        "EVAL_NUM_ENVS": EVAL_NUM_ENVS,
        "DIFFUSION_STEPS_EVAL": DIFFUSION_STEPS_EVAL,
        # Always disable W&B inside the notebook
        "USE_WANDB": False,
        "CHECKPOINT_PATH": DIFFUSION_ONLINE_CKPT,
    }
    run_inference(inference_cfg)


In [ ]:
# =============================================================================
# CELL 6 (PRIORITY) — visualise the diffusion planner acting in Craftax
# =============================================================================
#
# Re-runs the MPC + historical-inpainting loop on a small batch of envs and
# captures per-step rewards / achievements / actions for plotting.
# -----------------------------------------------------------------------------

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from src.diffusion.sampling import sample_plan_inpainting

VIZ_NUM_ENVS = 4
VIZ_STEPS = 600

if "Classic" not in ENV_NAME:
    print(f"Skipping behaviour viz: no diffusion checkpoint for {ENV_NAME!r}.")
else:
    viz_env = make_craftax_env_from_name(DIFFUSION_ENV_NAME, auto_reset=True)
    viz_params = viz_env.default_params

    rng = jax.random.PRNGKey(SEED + 1)
    rng, env_rng = jax.random.split(rng)
    obs0, state0 = jax.vmap(viz_env.reset, in_axes=(0, None))(
        jax.random.split(env_rng, VIZ_NUM_ENVS), viz_params,
    )
    history0 = jnp.full((VIZ_NUM_ENVS, PLAN_HORIZON), NUM_ACTIONS, dtype=jnp.int32)
    hist_len0 = jnp.zeros(VIZ_NUM_ENVS, dtype=jnp.int32)
    env_indices = jnp.arange(VIZ_NUM_ENVS)

    @jax.jit
    def viz_step(carry, _):
        obs, state, rng, history, hist_len = carry
        rng, plan_rng, env_rng = jax.random.split(rng, 3)

        # Reset history when plan window is exhausted (matches inference.py).
        seq_full = hist_len >= PLAN_HORIZON
        hist_len = jnp.where(seq_full, 0, hist_len)
        history = jnp.where(seq_full[:, None], NUM_ACTIONS, history)

        plan = sample_plan_inpainting(
            apply_eval, diffusion_params, plan_rng, obs,
            history, hist_len, NUM_ACTIONS, PLAN_HORIZON,
            DIFFUSION_STEPS_EVAL,
            ARCH_CFG["TEMPERATURE"], ARCH_CFG["TOP_P"],
        )
        action = jnp.take_along_axis(plan, hist_len[:, None], axis=-1).squeeze(-1)
        history = history.at[env_indices, hist_len].set(action)
        hist_len = hist_len + 1

        obs_next, state_next, reward, done, _ = jax.vmap(
            viz_env.step, in_axes=(0, 0, 0, None),
        )(jax.random.split(env_rng, VIZ_NUM_ENVS), state, action, viz_params)

        hist_len = jnp.where(done, 0, hist_len)
        history = jnp.where(done[:, None], NUM_ACTIONS, history)
        return (
            (obs_next, state_next, rng, history, hist_len),
            (action, reward, done, state_next.achievements),
        )

    print(f"Rolling out {VIZ_NUM_ENVS} agents for {VIZ_STEPS} steps...")
    _, (acts, rews, dones, achs) = jax.lax.scan(
        viz_step, (obs0, state0, rng, history0, hist_len0), jnp.arange(VIZ_STEPS),
    )
    acts_np = np.array(acts)              # [T, E]
    rews_np = np.array(rews)              # [T, E]
    achs_np = np.array(achs)              # [T, E, num_ach]
    dones_np = np.array(dones)            # [T, E]

    # First-life cumulative reward + unlock count per agent
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    cum_reward = np.cumsum(rews_np, axis=0)
    unlock_count = achs_np.sum(axis=-1)
    for e in range(VIZ_NUM_ENVS):
        end = np.where(dones_np[:, e])[0]
        cutoff = int(end[0]) + 1 if len(end) > 0 else VIZ_STEPS
        axes[0].plot(np.arange(cutoff), cum_reward[:cutoff, e], label=f"agent {e}")
        axes[1].plot(np.arange(cutoff), unlock_count[:cutoff, e], label=f"agent {e}")
    axes[0].set_title("Cumulative reward (first life)")
    axes[0].set_xlabel("env step"); axes[0].set_ylabel("cumulative reward")
    axes[1].set_title("Achievements unlocked")
    axes[1].set_xlabel("env step"); axes[1].set_ylabel("# unique achievements")
    for ax in axes:
        ax.legend(loc="best", fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # Action histogram (which actions does the planner actually use?)
    from craftax.craftax_classic.constants import Action as ClassicAction
    action_names = [a.name for a in ClassicAction]
    counts = np.bincount(acts_np.flatten(), minlength=NUM_ACTIONS)
    order = np.argsort(-counts)
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.bar(range(NUM_ACTIONS), counts[order])
    ax.set_xticks(range(NUM_ACTIONS))
    ax.set_xticklabels([action_names[i] for i in order], rotation=70, ha="right", fontsize=8)
    ax.set_title(f"Action usage over {VIZ_STEPS * VIZ_NUM_ENVS} env steps")
    ax.set_ylabel("count"); ax.grid(alpha=0.3, axis="y")
    plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# CELL 7 (PRIORITY) — visualise the ReMDM iterative unmasking process
# =============================================================================
#
# `sample_plan_inpainting` is JIT-compiled via lax.scan, so to capture the
# intermediate token sequences we re-implement its body as a Python loop. This
# is faithful to the production sampler — only the loop construct differs.
# -----------------------------------------------------------------------------

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

if "Classic" not in ENV_NAME:
    print(f"Skipping denoising viz: no diffusion checkpoint for {ENV_NAME!r}.")
else:
    DENOISE_STEPS = 12  # show enough steps to see iterative unmasking
    MASK_ID = NUM_ACTIONS

    # Sample one observation from the env to condition on.
    viz_env = make_craftax_env_from_name(DIFFUSION_ENV_NAME, auto_reset=True)
    viz_params = viz_env.default_params
    one_rng = jax.random.PRNGKey(SEED + 2)
    obs1, _state1 = viz_env.reset(one_rng, viz_params)
    obs_b = obs1[None, :]                      # [1, obs_dim]

    seq = jnp.full((1, PLAN_HORIZON), MASK_ID, dtype=jnp.int32)
    history = jnp.full((1, PLAN_HORIZON), MASK_ID, dtype=jnp.int32)
    hist_len = jnp.zeros((1,), dtype=jnp.int32)
    rng = jax.random.PRNGKey(SEED + 3)
    temperature = float(ARCH_CFG["TEMPERATURE"])
    top_p = float(ARCH_CFG["TOP_P"])

    trace = [np.array(seq[0])]               # row 0 = fully masked

    # Mirror the body of src.diffusion.sampling.sample_plan_inpainting._step
    for step in range(1, DENOISE_STEPS + 1):
        rng, model_rng, sample_rng, remask_rng = jax.random.split(rng, 4)
        ratio = step / DENOISE_STEPS
        t_tensor = jnp.full((1,), 1.0 - ratio)
        logits = apply_eval(diffusion_params, obs_b, seq, t_tensor, model_rng) / max(temperature, 1e-8)

        # Nucleus filtering
        probs = jax.nn.softmax(logits, axis=-1)
        sorted_idx = jnp.argsort(-probs, axis=-1)
        sorted_p = jnp.take_along_axis(probs, sorted_idx, axis=-1)
        cutoff = jnp.cumsum(sorted_p, axis=-1) - sorted_p
        inv_idx = jnp.argsort(sorted_idx, axis=-1)
        nucleus_mask = jnp.take_along_axis(cutoff >= top_p, inv_idx, axis=-1)
        logits = jnp.where(nucleus_mask, -jnp.inf, logits)

        preds = jax.random.categorical(sample_rng, logits, axis=-1)
        conf = jnp.take_along_axis(
            jax.nn.softmax(logits, axis=-1), preds[..., None], axis=-1,
        ).squeeze(-1)
        num_unmask = max(1, int(PLAN_HORIZON * ratio))
        sorted_conf = jnp.sort(conf, axis=-1)[..., ::-1]
        thresh = sorted_conf[0, num_unmask - 1]
        seq_new = jnp.where(conf < thresh, MASK_ID, preds)

        # ReMDM-style remasking (matches sample_plan_inpainting)
        remask_prob = 0.15 * (1.0 - ratio)
        do_remask = (
            (jax.random.uniform(remask_rng, seq_new.shape) < remask_prob)
            & (seq_new != MASK_ID)
        )
        seq_new = jnp.where(do_remask, MASK_ID, seq_new)

        # Lock historical prefix (here: empty, so no-op)
        pos = jnp.broadcast_to(jnp.arange(PLAN_HORIZON)[None, :], (1, PLAN_HORIZON))
        seq_new = jnp.where(pos < hist_len[:, None], history, seq_new)

        seq = seq_new
        trace.append(np.array(seq[0]))

    trace = np.stack(trace)                  # [steps+1, plan_horizon]

    # Heatmap: rows = denoising step, columns = action position
    # Mask cells coloured grey, action cells coloured by token id (viridis).
    masked = trace == MASK_ID
    fig, ax = plt.subplots(figsize=(12, 5))
    cmap = plt.cm.viridis.copy()
    display = np.where(masked, np.nan, trace.astype(float))
    im = ax.imshow(
        display, aspect="auto", cmap=cmap, vmin=0, vmax=NUM_ACTIONS - 1,
        interpolation="nearest",
    )
    # Overlay grey for masked cells
    ax.imshow(
        np.where(masked, 1.0, np.nan), aspect="auto",
        cmap=ListedColormap(["#dddddd"]), vmin=0, vmax=1, interpolation="nearest",
    )
    ax.set_xlabel("plan position (action token)")
    ax.set_ylabel("denoising step")
    ax.set_title(
        f"ReMDM iterative unmasking ({DENOISE_STEPS} steps, plan_horizon={PLAN_HORIZON})"
        "\nGrey = MASK token, colour = sampled action ID"
    )
    cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label("action ID")
    plt.tight_layout(); plt.show()

    n_masked = masked.sum(axis=1)
    print(f"Masked tokens per step: {list(n_masked)}")
    print(f"  step 0  (start): {int(n_masked[0])}/{PLAN_HORIZON} masked")
    print(f"  step {DENOISE_STEPS} (final): {int(n_masked[-1])}/{PLAN_HORIZON} masked")


In [ ]:
# =============================================================================
# CELL 8 — PPO expert baseline (loaded live, evaluated on `ENV_NAME`)
# =============================================================================
#
# DAgger trains the diffusion planner to imitate this expert. We expect the
# DAgger return (Cell 5) to be close to the PPO expert return on Classic.
# For Full Craftax (where no diffusion checkpoint exists), this is the only
# evaluation that can run.
# -----------------------------------------------------------------------------

import os
import numpy as np
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
from orbax.checkpoint import checkpoint_utils
from craftax.craftax_env import make_craftax_env_from_name

from src.planners.ppo import build_ppo_network, PPOAgent

PPO_NUM_ENVS = EVAL_NUM_ENVS
PPO_EVAL_STEPS = min(EVAL_STEPS, 1500)  # PPO eval is fast — bound it for snappy demo

ppo_path = os.path.abspath(PPO_CKPT[ENV_NAME])
ppo_env = make_craftax_env_from_name(ENV_NAME, auto_reset=True)
ppo_params_env = ppo_env.default_params
ppo_num_actions = int(ppo_env.action_space(ppo_params_env).n)
ppo_obs_dim = int(ppo_env.observation_space(ppo_params_env).shape[0])

# Build the PPO-RNN network and an abstract param pytree from a dummy init.
PPO_LAYER_SIZE = 512
ppo_net = build_ppo_network("ppo_rnn", ppo_num_actions, PPO_LAYER_SIZE,
                            {"LAYER_SIZE": PPO_LAYER_SIZE})
_dummy_x = (jnp.zeros((1, PPO_NUM_ENVS, ppo_obs_dim)),
            jnp.zeros((1, PPO_NUM_ENVS)))
_abstract_params = ppo_net.init(
    jax.random.PRNGKey(0),
    jnp.zeros((PPO_NUM_ENVS, PPO_LAYER_SIZE)),
    _dummy_x,
)

# Restore params only (the on-disk checkpoint also contains opt_state which we
# don't need for inference). `partial_restore=True` lets us read just `params`,
# and `construct_restore_args` is required on orbax >= 0.11 to provide sharding.
_restore_args = checkpoint_utils.construct_restore_args({"params": _abstract_params})
with ocp.CheckpointManager(ppo_path) as _mgr:
    _step = _mgr.latest_step()
    _restored = _mgr.restore(
        _step,
        args=ocp.args.PyTreeRestore(
            item={"params": _abstract_params},
            restore_args=_restore_args,
            partial_restore=True,
        ),
    )
print(f"Loaded PPO_RNN checkpoint from '{ppo_path}' (step {_step})")

ppo_agent = PPOAgent(
    network=ppo_net,
    params=_restored["params"],
    model_type="ppo_rnn",
    layer_size=PPO_LAYER_SIZE,
)

rng = jax.random.PRNGKey(SEED + 100)
rng, env_rng = jax.random.split(rng)
obs, state = jax.vmap(ppo_env.reset, in_axes=(0, None))(
    jax.random.split(env_rng, PPO_NUM_ENVS), ppo_params_env,
)
hidden0 = ppo_agent.init_hidden(PPO_NUM_ENVS)
done0 = jnp.zeros(PPO_NUM_ENVS, dtype=bool)


@jax.jit
def ppo_step(carry, _):
    obs, state, hidden, done, rng = carry
    rng, act_rng, env_rng = jax.random.split(rng, 3)
    action, hidden = ppo_agent.act(obs, done, hidden, act_rng, temperature=1.0)
    obs_next, state_next, reward, done_next, _ = jax.vmap(
        ppo_env.step, in_axes=(0, 0, 0, None),
    )(jax.random.split(env_rng, PPO_NUM_ENVS), state, action, ppo_params_env)
    return (obs_next, state_next, hidden, done_next, rng), (reward, done_next, state_next.achievements)


print(f"Running PPO expert: {PPO_NUM_ENVS} envs x {PPO_EVAL_STEPS} steps on {ENV_NAME}...")
_, (rewards, dones, achievements) = jax.lax.scan(
    ppo_step, (obs, state, hidden0, done0, rng), jnp.arange(PPO_EVAL_STEPS),
)

rewards_np = np.array(rewards)
dones_np = np.array(dones)
ach_np = np.array(achievements)

# First-life evaluation (matches src/planners/inference.py convention)
ep_returns = np.zeros(PPO_NUM_ENVS)
ep_unlocks = np.zeros(PPO_NUM_ENVS, dtype=int)
for i in range(PPO_NUM_ENVS):
    deaths = np.where(dones_np[:, i])[0]
    end = int(deaths[0]) if len(deaths) > 0 else PPO_EVAL_STEPS - 1
    ep_returns[i] = rewards_np[: end + 1, i].sum()
    ep_unlocks[i] = int(ach_np[: end + 1, i].max(axis=0).sum())

print()
print(f"PPO expert mean return : {ep_returns.mean():.2f}  (best={ep_returns.max():.2f})")
print(f"PPO expert mean unlocks: {ep_unlocks.mean():.2f} achievements")
if "Classic" in ENV_NAME:
    print()
    print(
        "Compare with the DAgger diffusion result printed in Cell 5. On the\n"
        "DAgger checkpoint with EVAL_STEPS=10000 / 32 envs, both methods score\n"
        "≈ 10.4 / 22 in the paper — i.e. DAgger has fully imitated the expert."
    )


## 2. RL fine-tuning ablation suite (pre-computed)

Once DAgger had reached the expert ceiling, we ran a **25-ablation suite** of
RL fine-tuning interventions trying to push past it. Every figure and table
below was generated offline by `experiments/rl_finetuning/run_ablations.py`
and shipped in the HF repo at
`experiments/rl_finetuning/outputs/craftax_classic_final_results/analysis/`.

The four groups tested in the ablation suite:

| Group | Hypothesis tested |
|---|---|
| **A** Regularisation | KL penalty, EWC, LLRD, LoRA, mixed replay, hard trust region |
| **B** Optimisation | t-curriculum, entropy bonus, PCGrad, advantage clip, normalised adv, BC-on-wins, low-t |
| **C** Capacity | head-only / FFN-only / attention-only / frozen backbone / top-k layer ablations |
| **D** Data | reward filtering, action diversity, running stats, learned reward model |

**Headline finding.** No ablation meaningfully improves on the DAgger
checkpoint (best Δ-score over baseline RL is +0.18 — within noise). Multiple
ablations *collapse* (Δ ≈ −10) — see the figures below.


In [ ]:
# =============================================================================
# CELL 10 — Pre-computed ablation figures
# =============================================================================

import os
from IPython.display import Image, display, Markdown

ABLATION_DIR = "experiments/rl_finetuning/outputs/craftax_classic_final_results/analysis/figures"

KEY_FIGURES = [
    (
        "group_comparison.png",
        "**Group comparison** — final score by ablation group. Group A "
        "(regularisation) clusters tightly around the baseline; Group C "
        "(capacity ablation) collapses.",
    ),
    (
        "score_delta_over_baseline_rl.png",
        "**Δ-score over baseline RL** — every ablation, sorted. The best "
        "improvement is < +0.2; many collapse to −10.",
    ),
    (
        "gradient_alignment.png",
        "**Gradient alignment** — cosine similarity between the BC and RL "
        "loss gradients during fine-tuning. Frequently negative, hinting at "
        "the underlying conflict.",
    ),
    (
        "gradient_conflict_map.png",
        "**Per-layer gradient conflict** — where in the network the BC and "
        "RL losses pull in opposite directions.",
    ),
]

for fname, caption in KEY_FIGURES:
    path = os.path.join(ABLATION_DIR, fname)
    if os.path.exists(path):
        display(Markdown(caption))
        display(Image(filename=path))
    else:
        display(Markdown(f"_Missing figure: `{fname}`_"))


In [ ]:
# =============================================================================
# CELL 11 — Ablation results tables
# =============================================================================

import os
import polars as pl

TABLES_DIR = "experiments/rl_finetuning/outputs/craftax_classic_final_results/analysis/tables"

main_df = pl.read_csv(os.path.join(TABLES_DIR, "main_results.csv")).sort(
    "Final_Score", descending=True,
)
print("Main ablation results (sorted by Final_Score, ceiling = 22):")
print(main_df)

verdict_df = pl.read_csv(os.path.join(TABLES_DIR, "hypothesis_verdict.csv"))
print()
print("Hypothesis verdicts:")
print(verdict_df.select(["Ablation", "Group", "Result", "Conclusion"]))


## 3. Conclusions

1. **DAgger works.** The online DAgger checkpoint imitates the PPO expert on
   Craftax-Classic to within noise (Cells 5 vs 8).
2. **RL fine-tuning does not help.** Across 25 ablations spanning
   regularisation, optimisation, capacity and data interventions, no method
   meaningfully improves on the DAgger checkpoint. The best Δ-score over
   baseline RL is +0.18, well within seed-to-seed variance.
3. **Several ablations actively collapse** (Group C: capacity restrictions,
   plus a handful of regularisers). The collapse pattern correlates with deep
   gradient flow into the backbone — see `gradient_conflict_map.png`.
4. **Framework independence.** The same finding holds in our parallel MiniHack
   PyTorch codebase, suggesting the obstruction is fundamental to RL
   fine-tuning of masked discrete diffusion planners rather than a Craftax-
   or JAX-specific quirk.

The full project documentation lives in the HF repo's `README.md`.
